In [0]:
%pip install fitparse

In [0]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.signal import savgol_filter

from utils.triathlon_utils import (
    parse_fit, 
    clean_fit_data, 
    calculate_elapsed_time, 
    apply_triathlon_style,
    format_time_axis,
    export_for_instagram,
    KLEUREN_PALETTE
)

from utils.config import DEFAULT_ATHLETE, DEFAULT_PATHS

In [0]:
def create_cardiac_drift_chart(
    df: pd.DataFrame,
    speed_threshold_kmh: float = 0.5,   # Drempel voor stilstand (<0.5 km/u)
    min_pause_sec: int = 150,           # Pauzes vanaf 2,5 minuut uitsluiten
    savgol_window: int = 301,           # 5 minuten venster voor smoothing
    savgol_polyorder: int = 2
):
    df_clean = df.copy()

    # --- 0. DYNAMISCHE KOLOMDETECTIE ---
    if 'time' in df_clean.columns:
        time_col = 'time'
    elif 'timestamp' in df_clean.columns:
        time_col = 'timestamp'
    else:
        raise KeyError("Het DataFrame bevat geen 'time' or 'timestamp' kolom.")

    df_clean[time_col] = pd.to_datetime(df_clean[time_col])
    df_clean = df_clean.sort_values(time_col).reset_index(drop=True)

    # Hartslagkolom
    hr_col = next((c for c in ['heart_rate', 'hr', 'heartrate'] if c in df_clean.columns), None)
    if hr_col is None:
        raise KeyError("Geen kolom gevonden voor hartslag.")

    # Hoogtekolom
    alt_col = next((c for c in ['altitude', 'enhanced_altitude', 'ele', 'elevation'] if c in df_clean.columns), None)

    # Snelheidskolom (in km/u)
    if 'speed_ms' in df_clean.columns:
        df_clean['raw_speed_kmh'] = df_clean['speed_ms'] * 3.6
    elif 'speed' in df_clean.columns:
        df_clean['raw_speed_kmh'] = df_clean['speed'] * 3.6
    elif 'enhanced_speed' in df_clean.columns:
        df_clean['raw_speed_kmh'] = df_clean['enhanced_speed'] * 3.6
    elif 'distance' in df_clean.columns:
        time_diff = df_clean[time_col].diff().dt.total_seconds().fillna(0)
        dist_diff = df_clean['distance'].diff().fillna(0)
        speed_ms = np.where(time_diff > 0, dist_diff / time_diff, 0)
        df_clean['raw_speed_kmh'] = speed_ms * 3.6
    else:
        raise KeyError("Geen kolom gevonden voor snelheid.")

    # --- 1. CONTINU TIJDSGRID HERCONSTRUEREN ---
    df_clean['time_diff_sec'] = df_clean[time_col].diff().dt.total_seconds().fillna(0)
    df_clean = df_clean.set_index(time_col)
    
    df_resampled = pd.DataFrame(index=pd.date_range(start=df_clean.index.min(), end=df_clean.index.max(), freq='1s'))
    df_resampled['speed_kmh'] = 0.0
    df_resampled['hr'] = np.nan

    for i in range(len(df_clean)):
        t_curr = df_clean.index[i]
        t_diff = df_clean['time_diff_sec'].iloc[i]
        spd = df_clean['raw_speed_kmh'].iloc[i]
        hr_val = df_clean[hr_col].iloc[i]
        
        if t_diff <= 15:
            t_prev = t_curr - pd.Timedelta(seconds=int(t_diff))
            df_resampled.loc[t_prev:t_curr, 'speed_kmh'] = spd
            df_resampled.loc[t_prev:t_curr, 'hr'] = hr_val
        else:
            df_resampled.loc[t_curr, 'speed_kmh'] = spd
            df_resampled.loc[t_curr, 'hr'] = hr_val

    df_resampled['hr'] = df_resampled['hr'].ffill().bfill()
    if alt_col is not None:
        df_resampled[alt_col] = df_clean[alt_col].resample('1s').ffill().bfill()

    df_resampled = df_resampled.reset_index().rename(columns={'index': time_col})

    # --- 2. PAUZEDETECTIE & ACTIEVE TIJD FILTEREN ---
    df_resampled['is_raw_idle'] = df_resampled['speed_kmh'] < speed_threshold_kmh
    df_resampled['pause_group'] = (~df_resampled['is_raw_idle']).cumsum()
    pause_durations = df_resampled[df_resampled['is_raw_idle']].groupby('pause_group')['is_raw_idle'].transform('count')
    df_resampled['is_valid_pause'] = df_resampled['is_raw_idle'] & (pause_durations >= min_pause_sec)

    # Filter pauzes eruit en bereken continue actieve tijd (in minuten)
    df_active = df_resampled[~df_resampled['is_valid_pause']].copy().reset_index(drop=True)
    df_active['active_min'] = np.arange(len(df_active)) / 60.0

    # --- 3. SMOOTHING OP ACTIEVE DATA ---
    if savgol_window % 2 == 0:
        savgol_window += 1
    actual_window = min(savgol_window, len(df_active) if len(df_active) % 2 != 0 else len(df_active) - 1)

    if len(df_active) >= actual_window and actual_window > savgol_polyorder:
        df_active['speed_kmh_smooth'] = savgol_filter(df_active['speed_kmh'], window_length=actual_window, polyorder=savgol_polyorder).clip(0, None)
        df_active['hr_smooth'] = savgol_filter(df_active['hr'], window_length=actual_window, polyorder=savgol_polyorder)
        if alt_col is not None:
            df_active['alt_smooth'] = savgol_filter(df_active[alt_col], window_length=actual_window, polyorder=savgol_polyorder)
    else:
        df_active['speed_kmh_smooth'] = df_active['speed_kmh']
        df_active['hr_smooth'] = df_active['hr']
        if alt_col is not None:
            df_active['alt_smooth'] = df_active[alt_col]

    # --- 4. CARDIAC DRIFT BEREKENING (HOOGTE-GECORRIGEERD) ---
    if alt_col is not None:
        alt_diff = df_active[alt_col].diff().fillna(0)
        climb_ms = np.where(alt_diff > 0, alt_diff, 0)
        # Gebruik de ruwe snelheid 'speed_kmh' voor een stabiele berekening
        df_active['equiv_speed_kmh_raw'] = df_active['speed_kmh'] + (climb_ms * 3.6 * 10)
    else:
        df_active['equiv_speed_kmh_raw'] = df_active['speed_kmh']

    half_point = len(df_active) // 2
    first_half = df_active.iloc[:half_point]
    second_half = df_active.iloc[half_point:]

    # Berekenen op ruwe hartslag en ruwe gecorrigeerde snelheid
    mean_hr_h1 = first_half['hr'].mean()
    mean_spd_h1 = first_half['equiv_speed_kmh_raw'].mean()
    
    mean_hr_h2 = second_half['hr'].mean()
    mean_spd_h2 = second_half['equiv_speed_kmh_raw'].mean()

    ef_h1 = mean_hr_h1 / mean_spd_h1 if mean_spd_h1 > 0 else 0
    ef_h2 = mean_hr_h2 / mean_spd_h2 if mean_spd_h2 > 0 else 0
    
    cardiac_drift_pct = ((ef_h2 - ef_h1) / ef_h1 * 100) if ef_h1 > 0 else 0.0

    # --- 5. GRAFIEK OPBOUWEN ---
    plt.style.use('dark_background')
    BG_COLOR = '#000000'
    CARD_BG = '#0D0D0D'
    ACCENT_CYAN = '#00F5FF'  # Snelheid (links)
    ACCENT_PINK = '#FF2A6D'  # Hartslag (rechts)

    fig = plt.figure(figsize=(14, 7), facecolor=BG_COLOR)
    gs = fig.add_gridspec(1, 2, width_ratios=[3.2, 1], wspace=0.20)

    # Zorg dat de figuur de achtergrondkleur regelt, en de subplots transparant zijn
    ax_main = fig.add_subplot(gs[0], facecolor='none')   # Snelheid (links)
    ax_hr = ax_main.twinx()                               # Hartslag (rechts)
    ax_donut = fig.add_subplot(gs[1], facecolor=BG_COLOR)

    # Ax 0: Hoogteprofiel als eerste op de achtergrond
    if alt_col is not None:
        ax_alt = ax_main.twinx()
        ax_alt.set_zorder(1)
        ax_main.set_zorder(2)
        ax_hr.set_zorder(3)
        ax_main.patch.set_visible(False)

        # Hoogtelijn en opvulling
        ax_alt.plot(df_active['active_min'], df_active['alt_smooth'], color='#4A5568', alpha=0.45, linewidth=1.2)
        ax_alt.fill_between(df_active['active_min'], df_active['alt_smooth'], color='#4A5568', alpha=0.22)
        ax_alt.axis('off')
        
        # Schaal uitrekken over de volledige achtergrond:
        alt_min = df_active['alt_smooth'].min()
        alt_max = df_active['alt_smooth'].max()
        alt_range = alt_max - alt_min if alt_max > alt_min else 100
        
        # Start strak op het dal en geef slechts een kleine marge aan de top
        ax_alt.set_ylim(alt_min - (alt_range * 0.05), alt_max + (alt_range * 0.15))

    # Ax 1: Snelheid (Linkeras - Cyan)
    ax_main.plot(
        df_active['active_min'], 
        df_active['speed_kmh_smooth'], 
        color=ACCENT_CYAN, 
        linewidth=1.8
    )
    ax_main.set_xlabel('Actieve Tijd (minuten)', color='white', fontsize=11, fontweight='bold', labelpad=8)
    ax_main.set_ylabel('Snelheid (km/u)', color=ACCENT_CYAN, fontsize=11, fontweight='bold', labelpad=8)
    ax_main.tick_params(colors=ACCENT_CYAN, labelsize=9)
    ax_main.grid(False)
    ax_main.set_ylim(bottom=0, top=max(df_active['speed_kmh_smooth'].max() * 1.15, 5))

    # Ax 2: Hartslag (Rechteras - Roze/Rood)
    ax_hr.plot(
        df_active['active_min'], 
        df_active['hr_smooth'], 
        color=ACCENT_PINK, 
        linewidth=1.8
    )
    ax_hr.set_ylabel('Hartslag (BPM)', color=ACCENT_PINK, fontsize=11, fontweight='bold', labelpad=8)
    ax_hr.tick_params(colors=ACCENT_PINK, labelsize=9)
    ax_hr.grid(False)
    ax_hr.set_ylim(bottom=df_active['hr_smooth'].min() * 0.85, top=df_active['hr_smooth'].max() * 1.10)

    ax_main.set_title('ETAPPE 5 PERFORMANCE: CARDIAC DRIFT & HOOGTE-IMPACT', color='white', fontsize=13, fontweight='bold', pad=15)

    # Stats Overlay Box
    stats_text = (
        f"Gem. HR (Helft 1): {mean_hr_h1:.0f} BPM\n"
        f"Gem. HR (Helft 2): {mean_hr_h2:.0f} BPM\n"
        f"Gecorr. Drift:   {cardiac_drift_pct:+.1f}%"
    )
    ax_main.text(
        0.02, 0.95, stats_text, 
        transform=ax_main.transAxes, 
        fontsize=10, color='white', fontweight='bold',
        verticalalignment='top',
        bbox=dict(boxstyle='round,pad=0.5', facecolor='#111827', alpha=0.85, edgecolor='#374151')
    )

    # --- DONUT CHART ---
    drift_val = max(0, min(100, abs(cardiac_drift_pct)))
    sizes = [drift_val, max(0, 100 - drift_val)]
    donut_color = ACCENT_PINK if cardiac_drift_pct > 5.0 else '#FFA500' if cardiac_drift_pct > 0 else ACCENT_CYAN

    ax_donut.pie(
        sizes, 
        colors=[donut_color, '#1A1A1A'], 
        startangle=90, 
        counterclock=False,
        wedgeprops=dict(width=0.26, edgecolor=BG_COLOR, linewidth=3)
    )

    ax_donut.text(
        0, 0, f"{cardiac_drift_pct:+.1f}%\nDrift", 
        ha='center', va='center', 
        fontsize=14, fontweight='bold', color='white'
    )
    ax_donut.set_title('Cardiac Drift %', color='white', fontsize=11, fontweight='bold', pad=12)

    fig.subplots_adjust(left=0.08, right=0.92, wspace=0.22)
    return fig, df_active

In [0]:
# Het FIT-bestand van etappe 5
fit_file_path = f"{DEFAULT_PATHS.fit_dir}/20260916_stubaier_hohenweg_etappe_5.fit"

# 1. Inlezen & Opschonen via de utils
df = parse_fit(fit_file_path)
df_clean = clean_fit_data(df)

# 2. Grafiek genereren
fig, df_processed = create_cardiac_drift_chart(
    df_clean,
    savgol_window=601,  # 10 min
    savgol_polyorder=1
)

# 3. Exporteer naar landscape formaat voor Instagram
export_for_instagram(fig, filename="20260916_cardiac_drift", format_type='landscape')